# 정규표현식 검토: 캡처·치환·과도한 매칭

공학프로그래밍입문 · 2026학년도 1학기 · 핵심 활동 약 40~50분

**목표는 문법 암기보다 원리를 이해하고, 코딩 AI에 정확한 질문·지시를 하며, 결과를 검토하는 능력입니다.**
코드는 예측한 뒤 실행하고, 값·조건을 직접 바꾸며 확인합니다. AI 도구를 사용할 수 없으면 제시된 가상 답변을 동료와 검토합니다.

준비: 정규표현식의 전체 매칭·그룹·경계

## 오늘 설명할 수 있어야 하는 것

- 그룹과 치환 결과를 원문과 대조한다.
- 탐욕적 매칭·경계 조건의 반례를 만들고 패턴 적용 한계를 설명한다.


## 1. 먼저 원리를 읽기

그룹은 필요한 부분을 추출하거나 치환에 재사용합니다. `.*`는 가능한 긴 범위, `.*?`는 가능한 짧은 범위를 찾습니다.
“짧게 매칭한다”만으로 복잡한 HTML을 올바르게 해석하는 것은 아닙니다. 아래에서는 형식이 단순하게 제한된 합성 지점 ID만 처리합니다.


## 2. 실행 전 예측하기

`"<b>A</b><b>B</b>"`에서 `<.*>`와 `<.*?>`는 각각 무엇을 찾을까요? 숫자 그룹만 바꾸면 구분자는 남을까요?


아래 기록을 먼저 채운 뒤 다음 코드 셀을 실행하세요. 정답과 다르면 예측에 사용한 가정을 고칩니다.

- 내 예상 결과:
- 그렇게 생각한 이유:
- 실행 뒤 발견한 차이:


In [ ]:
import re

text = "<b>A</b><b>B</b>"
greedy = re.findall(r"<.*>", text)
nongreedy = re.findall(r"<.*?>", text)
print("greedy:", greedy, "non-greedy:", nongreedy)
assert greedy == [text]
assert nongreedy == ["<b>", "</b>", "<b>", "</b>"]


### 관찰을 원리와 연결하기

두 패턴 모두 오류 없이 실행됩니다. 원하는 결과가 “태그”인지 “태그 안의 텍스트”인지에 따라 옳고 그름이 달라집니다. 복잡한 HTML은 파서가 필요한지 검토하세요.


다음 셀의 입력이나 조건을 하나 바꾸세요. 바꾼 값, 예상한 변화, 실제 결과를 적습니다. 단순 재실행만으로 이 활동을 마치지 않습니다.


In [ ]:
pattern = re.compile(r"(?<![A-Za-z0-9_-])(?P<prefix>ST-)[0-9]{3}(?![A-Za-z0-9_-])")
original = "A: ST-012, B: ST-003; ST-0123은 제외"
masked = pattern.sub(r"\g<prefix>***", original)
print(masked)
assert masked == "A: ST-***, B: ST-***; ST-0123은 제외"
# 연속된 ID 문자 일부만 치환하지 않도록 앞뒤 조건을 명시했습니다.


## 3. 작은 기능을 직접 완성하기

`mask_station_ids(text)`를 완성하세요. ASCII `ST-` + 숫자 3개를 `ST-***`로 치환합니다. 대소문자는 구분합니다.
앞뒤에 ASCII 영문자·숫자·밑줄·하이픈이 붙어 있으면 더 큰 토큰의 일부로 보고 치환하지 않습니다. 공백·쉼표·괄호는 허용 경계입니다. 나머지 글자·공백·줄바꿈은 그대로 보존합니다. 이 경계 규칙은 실습 계약이며 범용 익명화 규칙은 아닙니다.


먼저 입력·출력 예를 하나 더 직접 만드세요. 모르는 문법은 도움말이나 AI에 질문해도 됩니다. AI 코드 전체를 받기 전에 필요한 처리 단계를 자신의 말로 2~3줄 적으세요.

- 내가 정한 추가 입력과 예상 출력:
- 처리 단계:


In [ ]:
def mask_station_ids(text):
    # TODO: 경계 조건과 치환 후 원문 보존을 함께 확인하세요.
    raise NotImplementedError("부분 치환이 일어나지 않게 구현하세요")


### 공개 점검

아래 점검 셀을 실행합니다. **“미완료”는 통과가 아닙니다.** 위 함수를 구현한 뒤 다시 실행하세요.
실패하면 기대값을 바꾸기보다 입력·계약·구현 중 무엇이 다른지 조사합니다. 공개 시험만 통과했다고 모든 입력에 옳다는 뜻은 아닙니다.


In [ ]:
def check_answer(candidate):
    cases = [
        ("ST-012 ST-003", "ST-*** ST-***"),
        ("(ST-012),\nST-999", "(ST-***),\nST-***"),
        ("XST-012 ST-0123 ST-012_A ST-012-x", "XST-012 ST-0123 ST-012_A ST-012-x"),
        ("st-012 ST-１２３", "st-012 ST-１２３"), ("", ""),
    ]
    for text, expected in cases:
        result = candidate(text)
        assert result == expected, (text, result, expected)
        assert candidate(result) == result, "반복 치환으로 추가 변화가 없어야 합니다"
    print("공개 점검 통과: 다중 치환·경계·원문 보존·재실행")

try:
    check_answer(mask_station_ids)
except NotImplementedError:
    print("미완료: 위 TODO를 구현한 뒤 이 셀을 다시 실행하세요. 아직 점검을 통과하지 않았습니다.")


## 4. AI 제안을 검토하고 수정 지시하기

다음은 오류를 검토하기 위해 만든 **가상 AI 답변**입니다. 실제 AI 응답이나 정답이 아닙니다.
실행 전에 어떤 요구사항을 놓쳤는지 찾고, 반례가 그 문제를 어떻게 드러내는지 설명하세요.


In [ ]:
def ai_suggestion(text):
    return re.sub(r"ST-[0-9]{3}", "ST-***", text)

sample = "ST-0123 XST-012"
print("가상 제안:", ai_suggestion(sample))
assert ai_suggestion(sample) != sample


### 검토 질문

문법상 맞는 치환이라도 원하지 않는 부분까지 바꿀 수 있습니다. 치환하면 안 되는 예, 바뀌지 않아야 할 문자, 두 번 실행했을 때의 결과를 함께 검토하세요.


- 위 제안이 어긴 요구사항:
- 오류를 드러내는 입력과 기대 결과:
- 한 문장 수정 지시:


### 코딩 AI에게 보낼 요청 작성

치환 대상·대소문자·ASCII 범위·토큰 경계·보존할 문자를 정의한다. 변경 전후 문자열과 거절 반례, 반복 치환 시험을 요구하고 범용 개인정보 익명화로 확대 해석하지 않게 범위를 명시한다.


위 문장을 그대로 복사하는 대신 오늘의 실제 입력·기대값·실패 조건을 넣어 요청문을 완성하세요.
아래 틀로 **구현 전 계획**을 먼저 요청하고, 받은 코드에는 새 반례를 하나 추가합니다.

```text
목적과 환경: Python / Google Colab에서 ... 하려고 한다.
입력: 자료형·구조·단위는 ... 이며, 예시는 ... 이다.
출력: ... 형식으로 ... 를 반환해야 한다.
조건: ... 는 포함하고 ... 는 제외한다. 잘못된 입력은 ... 로 처리한다.
검증: 정상 예 ... / 경계 예 ... / 반례 ... 의 기대 결과는 ... 이다.
먼저 모호한 조건을 짚고 처리 계획을 설명해 줘. 그 뒤 짧은 코드와 검증을 제시해 줘.
```

- 내가 보낸 최초 요청:
- 검토 후 보낸 수정 요청:
- AI 제안에서 채택·수정·거절한 부분과 근거:
- 내가 추가한 시험의 입력·기대값·실제값:


## 5. 새 상황에 옮겨 보기

한글·URL·중첩 HTML이 섞인 로그에도 같은 규칙을 쓸 수 있을까요? 새 요구사항을 정하고 정규표현식 대신 파서를 선택할 상황을 제시하세요.


이 활동은 한 가지 구현을 암기했는지가 아니라 요구사항이 달라졌을 때 무엇을 다시 질문하는지 확인합니다.


## 마무리 기록

- 오늘 사용한 핵심 원리를 자신의 말로 한 문장:
- 결과를 신뢰할 근거(시험 입력·기대값·실제값):
- 아직 확인하지 않은 조건 또는 적용 한계:
- AI 없이 설명할 수 있는 코드 한 줄과 그 역할:

**완료 기준:** 예측을 남겼고, 입력·조건을 바꿨고, TODO를 구현해 공개 점검을 통과했고, 새 반례와 수정 지시를 작성했으며, 결과의 한계를 설명할 수 있습니다.

[과목의 실습 방식과 점검 기준](../LEARNING_GUIDE.md)을 참고하세요.


## 선택 확장

|는 두 패턴 중 하나를 고르는 선택 연산입니다. ^·$·단어 경계·lookaround가 소비하는 문자와 확인만 하는 위치를 구분하세요.


## 출처와 개편 범위

    기존 강의 노트북의 주제를 바탕으로 핵심 예제, 합성 연습 데이터, AI 검토 활동을 새로 구성했습니다.
    원본 파일명은 유지했고, 원본 Drive 링크와 개편 전·후 해시는 강의 폴더의 `materials-manifest.json`에 기록합니다.
    노트북 실행에 외부 AI API나 원본 로컬 ZIP은 필요하지 않습니다. 학습용 오류 제안은 반례를 확인하는 코드이며, 미완성 TODO와 구분합니다.
